# Model Improvement and Cross-Validation

This notebook compares regularized Random Forest and XGBoost
models using account-grouped cross-validation.

Primary evaluation metric: Brier score.

The existing 5% churn assumption and training-prevalence
baseline are retained as comparators.

The final test set will not be used during model selection.

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
    cross_validate
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score
)

from xgboost import XGBClassifier

df = pd.read_csv(
    "../data/raw/Case_Study_2_CLV_Predictive_Analytics.csv"
)

TARGET = "churn_event_next_12m"

labeled_df = df[df[TARGET].notna()].copy()

labeled_df["utilization_above_100_flag"] = (
    labeled_df["utilization_rate_pct"] > 1
).astype(int)

split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.30, random_state=42
)

train_idx, temp_idx = next(
    split1.split(
        labeled_df,
        groups=labeled_df["account_id"]
    )
)

train_df = labeled_df.iloc[train_idx].copy()
temp_df = labeled_df.iloc[temp_idx].copy()

split2 = GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=42
)

val_idx, test_idx = next(
    split2.split(
        temp_df,
        groups=temp_df["account_id"]
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

assert set(train_df.account_id).isdisjoint(val_df.account_id)
assert set(train_df.account_id).isdisjoint(test_df.account_id)
assert set(val_df.account_id).isdisjoint(test_df.account_id)

Train: 1328
Validation: 280
Test: 276


In [2]:
CORE_FEATURES = [
    "current_capacity_tb",
    "utilization_rate_pct",
    "utilization_above_100_flag",
    "expansion_potential_signal"
]

EXPANDED_FEATURES = CORE_FEATURES + [
    "historical_15yr_net_profit_usd",
    "contract_end_year",
    "contract_end_month"
]

for subset in [train_df, val_df]:
    dates = pd.to_datetime(
        subset["contract_end_date"]
    )
    subset["contract_end_year"] = dates.dt.year
    subset["contract_end_month"] = dates.dt.month

def make_preprocessor(features):
    numeric = [
        col for col in features
        if col != "expansion_potential_signal"
    ]

    return ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            ["expansion_potential_signal"]
        )
    ])

In [3]:
experiments = {
    "RF Core": (
        CORE_FEATURES,
        RandomForestClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )
    ),
    "XGB Core": (
        CORE_FEATURES,
        XGBClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    ),
    "RF Expanded": (
        EXPANDED_FEATURES,
        RandomForestClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        )
    ),
    "XGB Expanded": (
        EXPANDED_FEATURES,
        XGBClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    )
}

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_results = []

for name, (features, model) in experiments.items():

    pipeline = Pipeline([
        ("preprocessor", make_preprocessor(features)),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        train_df[features],
        train_df[TARGET].astype(int),
        groups=train_df["account_id"],
        cv=cv,
        scoring={
            "brier": "neg_brier_score",
            "roc_auc": "roc_auc",
            "log_loss": "neg_log_loss"
        },
        n_jobs=1
    )

    cv_results.append({
        "Experiment": name,
        "CV Brier": -scores["test_brier"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean(),
        "CV Log Loss": -scores["test_log_loss"].mean(),
        "Brier SD": scores["test_brier"].std()
    })

cv_table = pd.DataFrame(cv_results)
display(cv_table.sort_values("CV Brier").round(4))

,Experiment,CV Brier,CV ROC-AUC,CV Log Loss,Brier SD
0,RF Core,0.1369,0.4656,0.4486,0.0009
2,RF Expanded,0.1372,0.4389,0.4491,0.0008
1,XGB Core,0.1378,0.4742,0.4524,0.0014
3,XGB Expanded,0.1389,0.4545,0.4556,0.0024


In [4]:
from sklearn.dummy import DummyClassifier

baseline_pipeline = DummyClassifier(
    strategy="prior"
)

baseline_scores = cross_validate(
    baseline_pipeline,
    train_df[CORE_FEATURES],
    train_df[TARGET].astype(int),
    groups=train_df["account_id"],
    cv=cv,
    scoring={
        "brier": "neg_brier_score",
        "roc_auc": "roc_auc",
        "log_loss": "neg_log_loss"
    }
)

print(
    "Training-prevalence CV Brier:",
    round(-baseline_scores["test_brier"].mean(), 4)
)

print(
    "Training-prevalence CV Log Loss:",
    round(-baseline_scores["test_log_loss"].mean(), 4)
)

Training-prevalence CV Brier: 0.1352
Training-prevalence CV Log Loss: 0.4416
